# ML Baseline Models: Regularized Variants

This notebook is a copied/derived baseline notebook for fixed follow-up experiments. It keeps the same `image_manifest.pkl` sample IDs and the same 30 x 26 source columns used by the CNN, but compares two numeric representations:

- `raw`: the current flattened/sequence numeric window.
- `window_norm`: the same window with price-like columns converted to within-window return-style values and indicators converted to within-window z-scores.

No hyperparameter search is performed. All model settings are fixed constants in the setup cell.

## 1. Setup

In [ ]:
from pathlib import Path
import sys
import random
import time
import copy
import json

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, f1_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "configs" / "cnnfin_1h.yaml").exists():
            return candidate
    raise FileNotFoundError("Could not find repo root containing configs/cnnfin_1h.yaml")


ROOT = find_repo_root(Path.cwd().resolve())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from cnnfin.config import load_config
from cnnfin.metrics import LABELS, bootstrap_macro_f1_ci, evaluate_predictions
from cnnfin.utils import ensure_dir, set_global_seed, write_json

CONFIG_PATH = ROOT / "configs" / "cnnfin_1h.yaml"
base_config = load_config(CONFIG_PATH)
artifact_dir = Path(base_config.artifact_dir)
if not artifact_dir.is_absolute():
    artifact_dir = ROOT / artifact_dir
config = load_config(CONFIG_PATH, artifact_dir=str(artifact_dir))

ARTIFACT_DIR = Path(config.artifact_dir)
PROCESSED_DIR = ARTIFACT_DIR / "processed"
RESULTS_DIR = ARTIFACT_DIR / "results"
MODEL_DIR = ARTIFACT_DIR / "models"
MERGED_PATH = PROCESSED_DIR / "merged_df.pkl"
FEATURE_COLUMNS_PATH = PROCESSED_DIR / "feature_columns.json"
IMAGE_MANIFEST_PATH = PROCESSED_DIR / "image_manifest.pkl"
SUMMARY_PATH = RESULTS_DIR / "ml_model_regularized_variant_summary.pkl"

# Set True for a quick smoke test only. Full Jarvis runs should keep this False.
DEBUG_MODE = False
DEBUG_ROWS_PER_CLASS = 32
DEBUG_NUM_EPOCHS = 1
DEBUG_BOOTSTRAP_ITERATIONS = 50

# Fixed follow-up experiment settings. These are not tuned in this notebook.
MAX_CLASS_WEIGHT = 10.0
VARIANTS = ["raw", "window_norm"]

XGB_PARAMS = {
    "objective": "multi:softprob",
    "num_class": len(LABELS),
    "eval_metric": "mlogloss",
    "max_depth": 3,
    "eta": 0.03,
    "subsample": 0.8,
    "colsample_bytree": 0.6,
    "lambda": 10.0,
    "min_child_weight": 25.0,
    "tree_method": "hist",
    "seed": int(config.seeds[0]),
    "nthread": -1,
}
XGB_NUM_BOOST_ROUND = 1000
XGB_EARLY_STOPPING_ROUNDS = 50

MLP_HIDDEN_DIMS = (128, 64)
MLP_DROPOUT = 0.4
LSTM_HIDDEN_SIZE = 64
LSTM_INPUT_DROPOUT = 0.2
LSTM_HEAD_DROPOUT = 0.4
TORCH_WEIGHT_DECAY = 1e-3

SEED = int(config.seeds[0])
set_global_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True


def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

DEVICE = get_device()
PIN_MEMORY = DEVICE.type == "cuda"
NUM_WORKERS = int(config.num_workers)
NUM_EPOCHS = DEBUG_NUM_EPOCHS if DEBUG_MODE else int(config.num_epochs)
BOOTSTRAP_ITERATIONS = DEBUG_BOOTSTRAP_ITERATIONS if DEBUG_MODE else int(config.bootstrap_iterations)

ensure_dir(RESULTS_DIR)
ensure_dir(MODEL_DIR)

print(f"Repo root: {ROOT}")
print(f"Config: {CONFIG_PATH}")
print(f"Artifact dir: {ARTIFACT_DIR}")
print(f"Merged df: {MERGED_PATH}")
print(f"Feature columns: {FEATURE_COLUMNS_PATH}")
print(f"Image manifest: {IMAGE_MANIFEST_PATH}")
print(f"Device: {DEVICE}")
print(f"Batch size: {config.batch_size}")
print(f"Epochs: {NUM_EPOCHS}")
print(f"Bootstrap iterations: {BOOTSTRAP_ITERATIONS}")
print(f"DEBUG_MODE: {DEBUG_MODE}")
print(f"MAX_CLASS_WEIGHT: {MAX_CLASS_WEIGHT}")
print(f"VARIANTS: {VARIANTS}")

## 2. Load CNN-Matched Samples And Source Features

In [ ]:
for required_path in [MERGED_PATH, FEATURE_COLUMNS_PATH, IMAGE_MANIFEST_PATH]:
    if not required_path.exists():
        if required_path == IMAGE_MANIFEST_PATH:
            raise FileNotFoundError(
                f"Missing {IMAGE_MANIFEST_PATH}. Run exploration/image_builder.ipynb with RUN_FULL_IMAGE_BUILD = True first."
            )
        raise FileNotFoundError(f"Missing required artifact: {required_path}")

merged_df = pd.read_pickle(MERGED_PATH).copy()
merged_df["Open time"] = pd.to_datetime(merged_df["Open time"], utc=True)
manifest = pd.read_pickle(IMAGE_MANIFEST_PATH).copy()
manifest["Open time"] = pd.to_datetime(manifest["Open time"], utc=True)
feature_columns = json.loads(FEATURE_COLUMNS_PATH.read_text())

source_cols = feature_columns["image_source_feature_cols"]
lookback = int(feature_columns["model_window_lookback"])
expected_dim = lookback * len(source_cols)

required_manifest_cols = {"sample_id", "Open time", "row_idx", "split", "label"}
missing_manifest_cols = sorted(required_manifest_cols - set(manifest.columns))
if missing_manifest_cols:
    raise ValueError(f"image_manifest.pkl is missing required columns: {missing_manifest_cols}")

missing_source_cols = [col for col in source_cols if col not in merged_df.columns]
if missing_source_cols:
    raise ValueError(f"merged_df.pkl is missing source feature columns: {missing_source_cols}")

manifest["label"] = manifest["label"].astype(int)
manifest["row_idx"] = manifest["row_idx"].astype(int)
manifest = manifest.sort_values(["Open time", "sample_id"]).reset_index(drop=True)

splits = {
    split: manifest[manifest["split"].eq(split)].copy().reset_index(drop=True)
    for split in ["train", "val", "test"]
}

if DEBUG_MODE:
    def debug_limit(frame: pd.DataFrame) -> pd.DataFrame:
        parts = []
        for label in LABELS:
            part = frame[frame["label"].eq(label)].head(DEBUG_ROWS_PER_CLASS)
            if len(part) == 0:
                raise ValueError(f"DEBUG_MODE requested label {label}, but split has no rows for it.")
            parts.append(part)
        return pd.concat(parts, ignore_index=True).sample(frac=1.0, random_state=SEED).reset_index(drop=True)

    splits = {split: debug_limit(frame) for split, frame in splits.items()}
    print("DEBUG_MODE is True: using balanced limited samples per split.")

summary_rows = []
for split, frame in splits.items():
    if len(frame) == 0:
        raise ValueError(f"Split {split} is empty.")
    label_counts = frame["label"].value_counts().sort_index().to_dict()
    summary_rows.append({"split": split, "rows": len(frame), **{f"label_{k}": label_counts.get(k, 0) for k in LABELS}})

print(f"Image-equivalent numeric window: {lookback} candles x {len(source_cols)} features = {expected_dim} tabular features")
print("Source columns:")
print(source_cols)
display(pd.DataFrame(summary_rows))

## 3. Build Raw And Window-Normalized Numeric Arrays

In [ ]:
def build_sequence_array(features: pd.DataFrame, samples: pd.DataFrame, columns: list[str], window: int) -> np.ndarray:
    values = features[columns].to_numpy(dtype=np.float32, copy=False)
    row_idx = samples["row_idx"].astype(int).to_numpy()
    out = np.empty((len(samples), window, len(columns)), dtype=np.float32)
    for i, end in enumerate(row_idx):
        start = int(end) - window + 1
        if start < 0:
            raise ValueError(f"Sample {samples.iloc[i]['sample_id']} has row_idx={end}, shorter than lookback={window}")
        out[i] = values[start : int(end) + 1]
    return out


def safe_divide_return(values: np.ndarray, anchor: np.ndarray, eps: float = 1e-12) -> np.ndarray:
    valid_anchor = np.where(np.isfinite(anchor) & (np.abs(anchor) > eps), anchor, np.nan)
    return values / valid_anchor - 1.0


def build_window_normalized_sequence(x_seq: np.ndarray, columns: list[str]) -> np.ndarray:
    raw = np.asarray(x_seq, dtype=np.float32)
    out = raw.copy()
    btc_ohlc_cols = [col for col in ["Open", "High", "Low", "Close"] if col in columns]
    btc_ohlc_idx = [columns.index(col) for col in btc_ohlc_cols]
    alt_close_idx = [i for i, col in enumerate(columns) if col.endswith("_Close")]
    price_like_idx = set(btc_ohlc_idx + alt_close_idx)
    indicator_idx = [i for i in range(len(columns)) if i not in price_like_idx]

    if "Close" in columns:
        close_idx = columns.index("Close")
        btc_anchor = raw[:, -1:, close_idx : close_idx + 1]
        for idx in btc_ohlc_idx:
            out[:, :, idx : idx + 1] = safe_divide_return(raw[:, :, idx : idx + 1], btc_anchor)

    for idx in alt_close_idx:
        anchor = raw[:, -1:, idx : idx + 1]
        out[:, :, idx : idx + 1] = safe_divide_return(raw[:, :, idx : idx + 1], anchor)

    if indicator_idx:
        indicator_values = raw[:, :, indicator_idx]
        means = np.nanmean(indicator_values, axis=1, keepdims=True)
        stds = np.nanstd(indicator_values, axis=1, keepdims=True)
        stds = np.where(np.isfinite(stds) & (stds > 1e-8), stds, np.nan)
        out[:, :, indicator_idx] = (indicator_values - means) / stds

    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


start_time = time.time()
X_seq_raw = {split: build_sequence_array(merged_df, frame, source_cols, lookback) for split, frame in splits.items()}
X_seq_variants = {
    "raw": X_seq_raw,
    "window_norm": {split: build_window_normalized_sequence(x_seq, source_cols) for split, x_seq in X_seq_raw.items()},
}
X_tab_variants = {
    variant: {split: X_seq_variants[variant][split].reshape(len(splits[split]), -1) for split in splits}
    for variant in VARIANTS
}
y = {split: splits[split]["label"].astype(int).to_numpy() for split in splits}

for variant in VARIANTS:
    print(f"\nVariant: {variant}")
    for split in ["train", "val", "test"]:
        x_seq = X_seq_variants[variant][split]
        x_tab = X_tab_variants[variant][split]
        finite_pct = float(np.isfinite(x_seq).mean() * 100.0)
        print(f"  {split}: X_seq={x_seq.shape}, X_tab={x_tab.shape}, y={y[split].shape}, finite={finite_pct:.2f}%")
print(f"Built numeric arrays in {time.time() - start_time:.1f}s")

## 4. Shared Evaluation And Weight Helpers

In [ ]:
def balanced_class_weights_np(y_train: np.ndarray) -> np.ndarray:
    counts = np.bincount(y_train.astype(int), minlength=len(LABELS)).astype(np.float32)
    total = counts.sum()
    weights = np.ones(len(LABELS), dtype=np.float32)
    for cls in LABELS:
        if counts[cls] > 0:
            weights[cls] = total / (len(LABELS) * counts[cls])
    return weights


def prediction_frame(samples: pd.DataFrame, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray | None) -> pd.DataFrame:
    out = samples[["sample_id", "Open time", "split", "row_idx"]].copy().reset_index(drop=True)
    out["y_true"] = y_true.astype(int)
    out["y_pred"] = y_pred.astype(int)
    if y_proba is not None:
        for cls in LABELS:
            out[f"p_{cls}"] = y_proba[:, cls]
    return out


def save_outputs(model_name: str, split: str, predictions: pd.DataFrame, metrics: dict) -> None:
    out_dir = ensure_dir(RESULTS_DIR / model_name)
    predictions.to_pickle(out_dir / f"{split}_predictions.pkl")
    write_json(metrics, out_dir / f"{split}_metrics.json")
    if split == "test":
        cm = pd.DataFrame(
            metrics["confusion_matrix"],
            index=[config.class_names[i] for i in LABELS],
            columns=[config.class_names[i] for i in LABELS],
        )
        cm.to_pickle(out_dir / "test_confusion_matrix.pkl")


def evaluate_and_save_model(model_name: str, split: str, y_true: np.ndarray, y_pred: np.ndarray, y_proba: np.ndarray | None) -> dict:
    metrics = evaluate_predictions(y_true, y_pred, class_names=config.class_names, y_proba=y_proba)
    metrics["macro_f1_ci"] = bootstrap_macro_f1_ci(
        y_true,
        y_pred,
        iterations=BOOTSTRAP_ITERATIONS if split == "test" else min(200, BOOTSTRAP_ITERATIONS),
        seed=SEED,
    )
    preds = prediction_frame(splits[split], y_true, y_pred, y_proba)
    save_outputs(model_name, split, preds, metrics)
    return metrics


def display_test_confusion(model_name: str, metrics: dict) -> None:
    cm = np.asarray(metrics["confusion_matrix"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=[config.class_names[i] for i in LABELS])
    fig, ax = plt.subplots(figsize=(4.5, 4.5))
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    plt.title(f"{model_name} Test Confusion Matrix")
    plt.show()


raw_class_weights = balanced_class_weights_np(y["train"])
capped_class_weights = np.minimum(raw_class_weights, MAX_CLASS_WEIGHT).astype(np.float32)
class_weight_dict = {int(i): float(w) for i, w in enumerate(capped_class_weights)}
raw_class_weight_dict = {int(i): float(w) for i, w in enumerate(raw_class_weights)}

print("Raw balanced class weights:", raw_class_weight_dict)
print("Capped class weights:", class_weight_dict)
if max(class_weight_dict.values()) > MAX_CLASS_WEIGHT:
    raise RuntimeError("Capped class weights exceed MAX_CLASS_WEIGHT.")

## 5. Logistic Regression

In [ ]:
for variant in VARIANTS:
    model_name = f"logistic_regression_{variant}"
    print(f"\nTraining {model_name}")
    start = time.time()
    X_tab = X_tab_variants[variant]
    logistic_model = Pipeline(
        [
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
            (
                "model",
                LogisticRegression(
                    max_iter=3000,
                    class_weight=class_weight_dict,
                    random_state=SEED,
                    n_jobs=-1,
                ),
            ),
        ]
    )
    logistic_model.fit(X_tab["train"], y["train"])
    joblib.dump(logistic_model, MODEL_DIR / f"{model_name}.joblib")

    val_pred = logistic_model.predict(X_tab["val"]).astype(int)
    val_proba = logistic_model.predict_proba(X_tab["val"])
    test_pred = logistic_model.predict(X_tab["test"]).astype(int)
    test_proba = logistic_model.predict_proba(X_tab["test"])

    val_metrics = evaluate_and_save_model(model_name, "val", y["val"], val_pred, val_proba)
    test_metrics = evaluate_and_save_model(model_name, "test", y["test"], test_pred, test_proba)
    print(f"{model_name} val_macro_f1={val_metrics['macro_f1']:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} elapsed={time.time() - start:.1f}s")
    display_test_confusion(model_name, test_metrics)

## 6. XGBoost

In [ ]:
try:
    import xgboost as xgb
except Exception as exc:
    raise RuntimeError("xgboost is required for this notebook.") from exc


def xgb_macro_f1_metric(predt: np.ndarray, dmatrix) -> tuple[str, float]:
    labels_true = dmatrix.get_label().astype(int)
    probs = predt
    if probs.ndim == 1:
        probs = probs.reshape(labels_true.shape[0], len(LABELS))
    labels_pred = probs.argmax(axis=1).astype(int)
    score = f1_score(labels_true, labels_pred, labels=LABELS, average="macro", zero_division=0)
    return "macro_f1", float(score)


def predict_xgb(booster, dmatrix) -> np.ndarray:
    best_iteration = getattr(booster, "best_iteration", None)
    if best_iteration is not None and best_iteration >= 0:
        return booster.predict(dmatrix, iteration_range=(0, best_iteration + 1))
    return booster.predict(dmatrix)


for variant in VARIANTS:
    model_name = f"xgboost_{variant}"
    print(f"\nTraining {model_name}")
    start = time.time()
    X_tab = X_tab_variants[variant]
    xgb_imputer = SimpleImputer(strategy="median")
    Xgb_train = xgb_imputer.fit_transform(X_tab["train"])
    Xgb_val = xgb_imputer.transform(X_tab["val"])
    Xgb_test = xgb_imputer.transform(X_tab["test"])
    sample_weight = np.asarray([class_weight_dict[int(label)] for label in y["train"]], dtype=np.float32)

    dtrain = xgb.DMatrix(Xgb_train, label=y["train"], weight=sample_weight)
    dval = xgb.DMatrix(Xgb_val, label=y["val"])
    dtest = xgb.DMatrix(Xgb_test, label=y["test"])
    evals_result = {}
    booster = xgb.train(
        XGB_PARAMS,
        dtrain,
        num_boost_round=XGB_NUM_BOOST_ROUND,
        evals=[(dval, "val")],
        custom_metric=xgb_macro_f1_metric,
        maximize=True,
        early_stopping_rounds=XGB_EARLY_STOPPING_ROUNDS,
        evals_result=evals_result,
        verbose_eval=False,
    )

    booster.save_model(MODEL_DIR / f"{model_name}.json")
    joblib.dump(
        {
            "imputer": xgb_imputer,
            "params": XGB_PARAMS,
            "num_boost_round": XGB_NUM_BOOST_ROUND,
            "early_stopping_rounds": XGB_EARLY_STOPPING_ROUNDS,
            "variant": variant,
            "source_cols": source_cols,
            "lookback": lookback,
            "evals_result": evals_result,
        },
        MODEL_DIR / f"{model_name}_preprocess.joblib",
    )

    val_proba = predict_xgb(booster, dval)
    test_proba = predict_xgb(booster, dtest)
    val_pred = val_proba.argmax(axis=1).astype(int)
    test_pred = test_proba.argmax(axis=1).astype(int)

    val_metrics = evaluate_and_save_model(model_name, "val", y["val"], val_pred, val_proba)
    test_metrics = evaluate_and_save_model(model_name, "test", y["test"], test_pred, test_proba)
    print(
        f"{model_name} best_iteration={getattr(booster, 'best_iteration', None)} "
        f"val_macro_f1={val_metrics['macro_f1']:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} "
        f"elapsed={time.time() - start:.1f}s"
    )
    display_test_confusion(model_name, test_metrics)

## 7. Shared PyTorch Training Helpers

In [ ]:
class ArrayDataset(Dataset):
    def __init__(self, x_array: np.ndarray, y_array: np.ndarray):
        self.x = torch.from_numpy(np.asarray(x_array, dtype=np.float32))
        self.y = torch.from_numpy(np.asarray(y_array, dtype=np.int64))

    def __len__(self) -> int:
        return len(self.y)

    def __getitem__(self, idx: int):
        return self.x[idx], self.y[idx]


def make_loader(x_array: np.ndarray, y_array: np.ndarray, *, shuffle: bool) -> DataLoader:
    generator = torch.Generator()
    generator.manual_seed(SEED)
    kwargs = {
        "batch_size": int(config.batch_size),
        "shuffle": shuffle,
        "num_workers": NUM_WORKERS,
        "pin_memory": PIN_MEMORY,
        "generator": generator if shuffle else None,
    }
    if NUM_WORKERS > 0:
        kwargs["persistent_workers"] = True
    return DataLoader(ArrayDataset(x_array, y_array), **kwargs)


def train_one_epoch_torch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module, optimizer: torch.optim.Optimizer) -> float:
    model.train()
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        optimizer.zero_grad(set_to_none=True)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        loss.backward()
        optimizer.step()
        rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * rows
        total_rows += rows
    return total_loss / max(total_rows, 1)


@torch.no_grad()
def predict_torch(model: nn.Module, loader: DataLoader, loss_fn: nn.Module) -> tuple[np.ndarray, np.ndarray, np.ndarray, float]:
    model.eval()
    y_true_parts = []
    y_pred_parts = []
    y_proba_parts = []
    total_loss = 0.0
    total_rows = 0
    for xb, yb in loader:
        xb = xb.to(DEVICE, non_blocking=PIN_MEMORY)
        yb = yb.to(DEVICE, non_blocking=PIN_MEMORY)
        logits = model(xb)
        loss = loss_fn(logits, yb)
        proba = torch.softmax(logits, dim=1)
        y_true_parts.append(yb.cpu().numpy())
        y_proba = proba.cpu().numpy()
        y_proba_parts.append(y_proba)
        y_pred_parts.append(y_proba.argmax(axis=1))
        rows = int(yb.shape[0])
        total_loss += float(loss.detach().cpu()) * rows
        total_rows += rows
    return np.concatenate(y_true_parts), np.concatenate(y_pred_parts), np.concatenate(y_proba_parts), total_loss / max(total_rows, 1)


def copy_state_to_cpu(model: nn.Module) -> dict[str, torch.Tensor]:
    return {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}


def fit_torch_classifier(
    model_name: str,
    model: nn.Module,
    train_loader: DataLoader,
    val_loader: DataLoader,
    test_loader: DataLoader,
    *,
    variant: str,
):
    print(f"Training {model_name}")
    start = time.time()
    model = model.to(DEVICE)
    loss_fn = nn.CrossEntropyLoss(weight=torch.tensor(capped_class_weights, dtype=torch.float32, device=DEVICE))
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate, weight_decay=TORCH_WEIGHT_DECAY)
    best_state = None
    best_score = -1.0
    best_record = None
    bad_epochs = 0
    history_rows = []

    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss = train_one_epoch_torch(model, train_loader, loss_fn, optimizer)
        val_true, val_pred, val_proba, val_loss = predict_torch(model, val_loader, loss_fn)
        val_metrics = evaluate_predictions(val_true, val_pred, class_names=config.class_names, y_proba=val_proba)
        record = {
            "epoch": int(epoch),
            "train_loss": float(train_loss),
            "val_loss": float(val_loss),
            "val_macro_f1": float(val_metrics["macro_f1"]),
            "val_accuracy": float(val_metrics["accuracy"]),
        }
        history_rows.append(record)
        print(f"{model_name} epoch={epoch} train_loss={train_loss:.5f} val_loss={val_loss:.5f} val_macro_f1={record['val_macro_f1']:.5f}")
        if record["val_macro_f1"] > best_score:
            best_score = record["val_macro_f1"]
            best_state = copy_state_to_cpu(model)
            best_record = record.copy()
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= config.early_stopping_patience:
                print(f"Early stopping {model_name} after {bad_epochs} epochs without validation macro-F1 improvement.")
                break

    if best_state is None:
        raise RuntimeError(f"{model_name} did not produce a best checkpoint.")
    model.load_state_dict({k: v.to(DEVICE) for k, v in best_state.items()})

    torch.save(
        {
            "model_state": model.state_dict(),
            "config": config.to_dict(),
            "best_val_macro_f1": float(best_score),
            "best_record": best_record,
            "variant": variant,
            "source_cols": source_cols,
            "lookback": lookback,
            "class_weights_raw": raw_class_weight_dict,
            "class_weights_capped": class_weight_dict,
            "weight_decay": TORCH_WEIGHT_DECAY,
        },
        MODEL_DIR / f"{model_name}.pt",
    )
    history = pd.DataFrame(history_rows)
    out_dir = ensure_dir(RESULTS_DIR / model_name)
    history.to_pickle(out_dir / "training_history.pkl")

    for split, frame, loader in [("val", splits["val"], val_loader), ("test", splits["test"], test_loader)]:
        true, pred, proba, loss = predict_torch(model, loader, loss_fn)
        metrics = evaluate_predictions(true, pred, class_names=config.class_names, y_proba=proba)
        metrics["loss"] = float(loss)
        metrics["macro_f1_ci"] = bootstrap_macro_f1_ci(
            true,
            pred,
            iterations=BOOTSTRAP_ITERATIONS if split == "test" else min(200, BOOTSTRAP_ITERATIONS),
            seed=SEED,
        )
        preds = prediction_frame(frame, true, pred, proba)
        save_outputs(model_name, split, preds, metrics)
        if split == "test":
            test_metrics = metrics

    print(f"{model_name} best_val_macro_f1={best_score:.5f} test_macro_f1={test_metrics['macro_f1']:.5f} elapsed={time.time() - start:.1f}s")
    display(history)
    display_test_confusion(model_name, test_metrics)
    return model, history, test_metrics

## 8. MLP

In [ ]:
class TabularMLP(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        h1, h2 = MLP_HIDDEN_DIMS
        self.net = nn.Sequential(
            nn.Linear(input_dim, h1),
            nn.BatchNorm1d(h1),
            nn.ReLU(),
            nn.Dropout(MLP_DROPOUT),
            nn.Linear(h1, h2),
            nn.BatchNorm1d(h2),
            nn.ReLU(),
            nn.Dropout(MLP_DROPOUT),
            nn.Linear(h2, len(LABELS)),
        )

    def forward(self, x):
        return self.net(x)


for variant in VARIANTS:
    model_name = f"mlp_{variant}"
    print(f"\nPreparing {model_name}")
    X_tab = X_tab_variants[variant]
    mlp_imputer = SimpleImputer(strategy="median")
    mlp_scaler = StandardScaler()
    X_mlp_train = mlp_scaler.fit_transform(mlp_imputer.fit_transform(X_tab["train"])).astype(np.float32)
    X_mlp_val = mlp_scaler.transform(mlp_imputer.transform(X_tab["val"])).astype(np.float32)
    X_mlp_test = mlp_scaler.transform(mlp_imputer.transform(X_tab["test"])).astype(np.float32)
    joblib.dump(
        {
            "imputer": mlp_imputer,
            "scaler": mlp_scaler,
            "source_cols": source_cols,
            "lookback": lookback,
            "variant": variant,
            "hidden_dims": MLP_HIDDEN_DIMS,
            "dropout": MLP_DROPOUT,
            "weight_decay": TORCH_WEIGHT_DECAY,
        },
        MODEL_DIR / f"{model_name}_preprocess.joblib",
    )

    mlp_train_loader = make_loader(X_mlp_train, y["train"], shuffle=True)
    mlp_val_loader = make_loader(X_mlp_val, y["val"], shuffle=False)
    mlp_test_loader = make_loader(X_mlp_test, y["test"], shuffle=False)

    fit_torch_classifier(
        model_name,
        TabularMLP(input_dim=expected_dim),
        mlp_train_loader,
        mlp_val_loader,
        mlp_test_loader,
        variant=variant,
    )

## 9. LSTM

In [ ]:
class NumericLSTM(nn.Module):
    def __init__(self, input_dim: int):
        super().__init__()
        self.input_dropout = nn.Dropout(LSTM_INPUT_DROPOUT)
        self.lstm = nn.LSTM(input_dim, hidden_size=LSTM_HIDDEN_SIZE, num_layers=1, batch_first=True)
        self.head = nn.Sequential(
            nn.Dropout(LSTM_HEAD_DROPOUT),
            nn.Linear(LSTM_HIDDEN_SIZE, 32),
            nn.ReLU(),
            nn.Dropout(LSTM_HEAD_DROPOUT),
            nn.Linear(32, len(LABELS)),
        )

    def forward(self, x):
        x = self.input_dropout(x)
        out, _ = self.lstm(x)
        return self.head(out[:, -1, :])


def fit_sequence_preprocessor(x_train: np.ndarray):
    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()
    train_rows = x_train.reshape(-1, len(source_cols))
    scaler.fit(imputer.fit_transform(train_rows))
    return imputer, scaler


def transform_sequence_split(x_seq: np.ndarray, imputer: SimpleImputer, scaler: StandardScaler) -> np.ndarray:
    flat = x_seq.reshape(-1, len(source_cols))
    scaled = scaler.transform(imputer.transform(flat)).astype(np.float32)
    return scaled.reshape(x_seq.shape)


for variant in VARIANTS:
    model_name = f"numeric_lstm_{variant}"
    print(f"\nPreparing {model_name}")
    X_seq = X_seq_variants[variant]
    lstm_imputer, lstm_scaler = fit_sequence_preprocessor(X_seq["train"])
    X_lstm_train = transform_sequence_split(X_seq["train"], lstm_imputer, lstm_scaler)
    X_lstm_val = transform_sequence_split(X_seq["val"], lstm_imputer, lstm_scaler)
    X_lstm_test = transform_sequence_split(X_seq["test"], lstm_imputer, lstm_scaler)
    joblib.dump(
        {
            "imputer": lstm_imputer,
            "scaler": lstm_scaler,
            "source_cols": source_cols,
            "lookback": lookback,
            "variant": variant,
            "hidden_size": LSTM_HIDDEN_SIZE,
            "input_dropout": LSTM_INPUT_DROPOUT,
            "head_dropout": LSTM_HEAD_DROPOUT,
            "weight_decay": TORCH_WEIGHT_DECAY,
        },
        MODEL_DIR / f"{model_name}_preprocess.joblib",
    )

    lstm_train_loader = make_loader(X_lstm_train, y["train"], shuffle=True)
    lstm_val_loader = make_loader(X_lstm_val, y["val"], shuffle=False)
    lstm_test_loader = make_loader(X_lstm_test, y["test"], shuffle=False)

    fit_torch_classifier(
        model_name,
        NumericLSTM(input_dim=len(source_cols)),
        lstm_train_loader,
        lstm_val_loader,
        lstm_test_loader,
        variant=variant,
    )

## 10. Summary

In [ ]:
summary_rows = []
for variant in VARIANTS:
    for base_model in ["logistic_regression", "xgboost", "mlp", "numeric_lstm"]:
        model_name = f"{base_model}_{variant}"
        metrics_path = RESULTS_DIR / model_name / "test_metrics.json"
        val_metrics_path = RESULTS_DIR / model_name / "val_metrics.json"
        if not metrics_path.exists():
            continue
        test_metrics = json.loads(metrics_path.read_text())
        val_metrics = json.loads(val_metrics_path.read_text()) if val_metrics_path.exists() else {}
        summary_rows.append(
            {
                "variant": variant,
                "model": base_model,
                "artifact_model_name": model_name,
                "val_macro_f1": val_metrics.get("macro_f1"),
                "test_macro_f1": test_metrics.get("macro_f1"),
                "test_accuracy": test_metrics.get("accuracy"),
                "test_weighted_f1": test_metrics.get("weighted_f1"),
                "macro_f1_ci_low": test_metrics.get("macro_f1_ci", {}).get("low"),
                "macro_f1_ci_high": test_metrics.get("macro_f1_ci", {}).get("high"),
            }
        )

summary = pd.DataFrame(summary_rows).sort_values("test_macro_f1", ascending=False, na_position="last").reset_index(drop=True)
summary.to_pickle(SUMMARY_PATH)
display(summary)
print(f"Saved regularized variant ML model summary: {SUMMARY_PATH}")

## 11. Acceptance Checks

In [ ]:
expected_model_names = [
    f"{base_model}_{variant}"
    for variant in VARIANTS
    for base_model in ["logistic_regression", "xgboost", "mlp", "numeric_lstm"]
]
required = []
for model_name in expected_model_names:
    required.extend(
        [
            RESULTS_DIR / model_name / "val_predictions.pkl",
            RESULTS_DIR / model_name / "test_predictions.pkl",
            RESULTS_DIR / model_name / "val_metrics.json",
            RESULTS_DIR / model_name / "test_metrics.json",
            RESULTS_DIR / model_name / "test_confusion_matrix.pkl",
        ]
    )
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError("Missing expected output files:\n" + "\n".join(missing[:20]))

if len(summary) != len(expected_model_names):
    raise ValueError(f"Expected {len(expected_model_names)} summary rows, found {len(summary)}")

sample_counts = {split: int(len(frame)) for split, frame in splits.items()}
if any(count <= 0 for count in sample_counts.values()):
    raise ValueError(f"Invalid sample counts: {sample_counts}")

if max(class_weight_dict.values()) > MAX_CLASS_WEIGHT:
    raise ValueError(f"Class weights are not capped correctly: {class_weight_dict}")

print("Regularized variant ML notebook completed successfully.")
print(f"All models used the same samples from: {IMAGE_MANIFEST_PATH}")
print(f"All models used the same source columns from: {FEATURE_COLUMNS_PATH}")
print(f"Sample counts: {sample_counts}")
print(f"Capped class weights: {class_weight_dict}")
print(f"Summary rows: {len(summary)}")
print(f"Numeric input surface: {lookback} x {len(source_cols)} = {expected_dim}")